# KinshipForge: Age-Progressive Child Face Synthesis
**Author:** Manaswi Mendhekar | B.Tech CSE (AI), CSVTU Bhilai
**Environment:** Designed for Kaggle T4 GPU

This notebook contains the complete inference and evaluation pipeline extending the CVPR 2023 StyleGene framework. It implements 5 novel contributions: Frozen DNA Seed, LERP Bucket Blending, Gender-Biased Layer Fusion, Multi-Seed Selection, and a rebuilt 8.11 GB FFHQ Gene Pool.

**Root cause identified:** The e4e encoder residual accounts for ~75% of facial widening (vs ~25% from latent_avg). See `e4e_geometric_bias_research/FINAL_ROOT_CAUSE_REPORT.md` for full analysis.

## 1. Dependencies
Importing standard libraries and deep learning frameworks required for the StyleGene pipeline.

In [1]:
!pip install lpips scikit-image einops kornia huggingface_hub gradio insightface onnxruntime -q

import os, sys, subprocess, random, importlib
import torch
import numpy as np
from PIL import Image
import insightface
from insightface.app import FaceAnalysis

print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 15.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 87.5 MB/s eta 0:00:00:00:0100:01
✅ Dependencies installed


## 2. Environment Setup & Repository Cloning
Cloning the base StyleGene architecture (CVPR 2023) and setting up the local working directory.

In [2]:
if not os.path.exists('./StyleGene'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/CVI-SZU/StyleGene.git',
                    './StyleGene'], check=True)

sys.path.insert(0, './StyleGene')
os.chdir('./StyleGene')
print("✅ Repo ready")

✅ Repo ready


## 3. Checkpoint Downloads
Fetching the required pre-trained weights for e4e inversion, StyleGAN2, and the facial alignment predictors. Checkpoints are cached to `/tmp/ckpt/`.

In [3]:
import os
import bz2
import urllib.request
import pandas as pd
from huggingface_hub import hf_hub_download

os.makedirs("/tmp/ckpt", exist_ok=True)

# Remove old landmark files
for f in [
    "/tmp/ckpt/shape_predictor_68_face_landmarks.dat.bz2",
    "/tmp/ckpt/shape_predictor_68_face_landmarks.dat",
]:
    if os.path.exists(f):
        os.remove(f)

# Download everything except the landmark model
files = [
    "e4e_ffhq_encode.pt",
    "stylegan2-ffhq-config-f.pt",
    "stylegene_N18.ckpt",
    "res34_fair_align_multi_7_20190809.pt",
]

for f in files:
    if not os.path.exists(f"/tmp/ckpt/{f}"):
        print(f"Downloading {f}...")
        hf_hub_download(
            repo_id="wmpscc/StyleGene_CKPT",
            filename=f,
            local_dir="/tmp/ckpt",
        )
    else:
        print(f"✓ {f}")

# Download landmark model from dlib instead of Hugging Face
bz2_path = "/tmp/ckpt/shape_predictor_68_face_landmarks.dat.bz2"
dat_path = "/tmp/ckpt/shape_predictor_68_face_landmarks.dat"

print("Downloading shape_predictor_68_face_landmarks.dat.bz2 from dlib...")
urllib.request.urlretrieve(
    "https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2",
    bz2_path,
)

print("Decompressing...")
with bz2.open(bz2_path, "rb") as src, open(dat_path, "wb") as dst:
    dst.write(src.read())

print(f"✓ Landmarks: {os.path.getsize(dat_path)/1e6:.1f} MB")

# Dummy CSV
pd.DataFrame(
    columns=["file_id", "gender", "age", "race"]
).to_csv("/tmp/ckpt/dummy_ffhq_attributes.csv", index=False)

print("✅ Checkpoints ready")

e4e_ffhq_encode.pt:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

stylegan2-ffhq-config-f.pt:   0%|          | 0.00/133M [00:00<?, ?B/s]

stylegene_N18.ckpt:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

res34_fair_align_multi_7_20190809.pt:   0%|          | 0.00/85.3M [00:00<?, ?B/s]

Decompressing...
✓ Landmarks: 99.7 MB
✅ Checkpoints ready


## 4. Model Configuration & Rebuilt Gene Pool
Injecting the custom 8.11 GB rebuilt FFHQ Gene Pool and pre-trained checkpoints into the configuration file.

In [4]:
configs_content = '''
path_ckpt_e4e           = "/tmp/ckpt/e4e_ffhq_encode.pt"
path_ckpt_stylegan2     = "/tmp/ckpt/stylegan2-ffhq-config-f.pt"
path_ckpt_stylegene     = "/tmp/ckpt/stylegene_N18.ckpt"
path_ckpt_genepool      = "/kaggle/input/datasets/manaswimendhekar/stylegene-balanced-pool/pool_50samples.pkl"
path_dataset_ffhq       = ""
path_csv_ffhq_attritube = "/tmp/ckpt/dummy_ffhq_attributes.csv"
path_ckpt_fairface      = "/tmp/ckpt/res34_fair_align_multi_7_20190809.pt"
path_ckpt_landmark68    = "/tmp/ckpt/shape_predictor_68_face_landmarks.dat.bz2"
'''
with open('./configs.py', 'w') as f:
    f.write(configs_content)

import importlib, configs
importlib.reload(configs)
from configs import *
print(path_ckpt_landmark68)

/tmp/ckpt/shape_predictor_68_face_landmarks.dat.bz2


## 5. Architecture Patching (Core Contributions)
This section dynamically patches the default StyleGene API to integrate novel contributions:
1. **Frozen DNA Seed:** Fixes crossover weights across all 3 age stages.
2. **LERP Bucket Blending:** Generates intermediate age genes via linear interpolation.
3. **Gender-Biased Layer Fusion:** Replaces standard 50/50 layer split with 70/30 weighting.
4. **Multi-Seed Selection:** Runs seeds [42, 123, 256] and selects the best via LPIPS.



In [5]:
import os

# Write a clean api.py that doesn't execute at import time
# Write a clean gene_crossover_mutation.py containing ARCS scaling logic
new_crossover = '''import torch
from .data_util import face_class, face_shape
import random

# Measured geometric sensitivity (mean aspect ratio drift) from complete diagnostics ground-truth
REGION_SENSITIVITY_MAP = {
    \'head\': 0.0217,
    \'head***cheek\': 0.0008,
    \'head***chin\': 0.0196,
    \'head***ear\': 0.0086,
    \'head***ear***helix\': 0.0086,
    \'head***ear***lobule\': 0.0086,
    \'head***eye***botton lid\': 0.0038,
    \'head***eye***eyelashes\': 0.0038,
    \'head***eye***iris\': 0.0038,
    \'head***eye***pupil\': 0.0038,
    \'head***eye***sclera\': 0.0038,
    \'head***eye***tear duct\': 0.0038,
    \'head***eye***top lid\': 0.0126,
    \'head***eyebrow\': 0.0030,
    \'head***forehead\': 0.0130,
    \'head***frown\': 0.0130,
    \'head***hair\': 0.0152,
    \'head***hair***sideburns\': 0.0294,
    \'head***jaw\': 0.0370,
    \'head***moustache\': 0.0050,
    \'head***mouth***inferior lip\': 0.0193,
    \'head***mouth***oral comisure\': 0.0193,
    \'head***mouth***superior lip\': 0.0153,
    \'head***mouth***teeth\': 0.0193,
    \'head***neck\': 0.0432,
    \'head***nose\': 0.0216,
    \'head***nose***ala of nose\': 0.0216,
    \'head***nose***bridge\': 0.0118,
    \'head***nose***nose tip\': 0.0216,
    \'head***nose***nostril\': 0.0216,
    \'head***philtrum\': 0.0077,
    \'head***temple\': 0.0052,
    \'head***wrinkles\': 0.0050
}

_last_logged_params = None

def reparameterize(mu, logvar):
    """
    Reparameterization trick to sample from N(mu, var) from N(0,1).
    """
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return eps * std + mu

def mix(w18_F, w18_M, w18_syn, geometry_weight=0.7, texture_weight=0.5,
        child_gender='male'):
    if child_gender == 'male':
        gw = geometry_weight
    else:
        gw = 1.0 - geometry_weight
    for k in [8, 9, 10, 11]:
        w18_syn[:, k, :] = w18_F[:, k, :] * gw + w18_M[:, k, :] * (1.0 - gw)
    for k in [12, 13, 14, 15, 16, 17]:
        w18_syn[:, k, :] = w18_F[:, k, :] * texture_weight + w18_M[:, k, :] * (1.0 - texture_weight)
    return w18_syn

def fuse_latent(w2sub34, sub2w, w18_F, w18_M, random_fakes, fixed_gamma=0.47, fixed_eta=0.4,
                arcs_lambda=0.0, sensitivity_map=None, child_gender='male',
                geometry_weight=0.7, texture_weight=0.5):
    global _last_logged_params
    device = w18_F.device

    mu_F, var_F, sub34_F = w2sub34(w18_F)
    mu_M, var_M, sub34_M = w2sub34(w18_M)
    new_sub34 = torch.zeros_like(sub34_F, dtype=torch.float, device=device)

    if len(random_fakes) == 0:  # EXCEPTION HANDLER (No matching gene pool)
        random_fakes = [(mu_F.cpu(), var_F.cpu())] + [(mu_M.cpu(), var_M.cpu())]

    # Resolve sensitivity map
    s_map = sensitivity_map if sensitivity_map is not None else REGION_SENSITIVITY_MAP
    s_vals = list(s_map.values())
    s_min = min(s_vals) if s_vals else 0.0
    s_max = max(s_vals) if s_vals else 1.0
    s_range = s_max - s_min if s_max != s_min else 1.0

    # Compute region-specific gammas using ARCS linear scale mapping
    resolved_gammas = {}
    for name in face_class:
        if name == \'background\':
            continue
        s_val = s_map.get(name, 0.0)
        s_norm = (s_val - s_min) / s_range
        g_val = fixed_gamma * (1.0 - arcs_lambda * s_norm)
        resolved_gammas[name] = g_val

    # Print ARCS configuration on first call or when hyperparameters change
    current_params = (fixed_gamma, arcs_lambda)
    if current_params != _last_logged_params:
        print(f"[ARCS] Crossover Setup | gamma_base={fixed_gamma:.2f} | lambda={arcs_lambda:.2f}")
        for name in face_class:
            if name == \'background\':
                continue
            neat_parts = name.split(\'***\')
            neat_name = " -> ".join([p.capitalize() for p in neat_parts])
            neat_name = neat_name.replace("Head -> ", "")
            print(f"  {neat_name:<30} gamma={resolved_gammas[name]:.3f} (sens={s_map.get(name, 0.0):.4f})")
        _last_logged_params = current_params

    # region genetic variation weights
    weights = {}
    for name in face_class:
        g_val = resolved_gammas.get(name, fixed_gamma)
        weights[name] = (random.uniform(0, 1 - g_val), g_val)

    # select genetic regions
    cur_class = random.sample(face_class, int(len(face_class) * (1 - float(fixed_eta))))

    for i, classname in enumerate(face_class):
        if classname == \'background\':
            new_sub34[:, :, i, :] = reparameterize(mu_F[:, :, i, :], var_F[:, :, i, :])
            continue

        if classname in cur_class:  # corresponding to t = 0 in Eq.10
            fake_mu, fake_var = random.choice(random_fakes)
            w_i, b_i = weights[classname]
            new_sub34[:, :, i, :] = reparameterize(
                mu_F[:, :, i, :] * w_i + fake_mu[:, :, i, :].to(device) * b_i + mu_M[:, :, i, :] * (1 - w_i - b_i),
                var_F[:, :, i, :] * w_i + fake_var[:, :, i, :].to(device) * b_i + var_M[:, :, i, :] * (1 - w_i - b_i))
        else:  # corresponding to t = 1 in Eq.10
            fake_mu, fake_var = random.choice(random_fakes)
            fake_latent = reparameterize(fake_mu[:, :, i, :], fake_var[:, :, i, :]).to(device)
            var = fake_latent
            new_sub34[:, :, i, :] = new_sub34[:, :, i, :] + var
    w18_syn = sub2w(new_sub34)

    w18_syn = mix(w18_F, w18_M, w18_syn, geometry_weight=geometry_weight, texture_weight=texture_weight, child_gender=child_gender)

    return w18_syn
'''

new_api = '''import torch
import numpy as np
import torch.nn.functional as F
import random
from models.stylegan2.model import Generator
from models.encoders.psp_encoders import Encoder4Editing
from models.stylegene.model import MappingSub2W, MappingW2Sub
from models.stylegene.util import get_keys, requires_grad, load_img
from models.stylegene.gene_crossover_mutation import fuse_latent
from configs import path_ckpt_e4e, path_ckpt_stylegan2, path_ckpt_stylegene

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")


class AncestryTuple(tuple):
    """
    A lightweight tuple wrapper for (mu, var) that stores metadata
    about the parent ancestry of the sampled mutation vector.
    """
    def __new__(cls, mu, var, ancestry):
        obj = super().__new__(cls, (mu, var))
        obj.ancestry = ancestry
        return obj


class BrdasList(list):
    """
    A list wrapper that tracks and logs the actual ancestry selections
    made by the existing StyleGene latent fusion routine (fuse_latent)
    when it performs random.choice() across face regions.
    """
    def __init__(self, items):
        super().__init__(items)
        self.selections = []
        self.non_bg_classes = [
            'head', 'head***cheek', 'head***chin', 'head***ear', 'head***ear***helix',
            'head***ear***lobule', 'head***eye***botton lid', 'head***eye***eyelashes', 'head***eye***iris',
            'head***eye***pupil', 'head***eye***sclera', 'head***eye***tear duct', 'head***eye***top lid',
            'head***eyebrow', 'head***forehead', 'head***frown', 'head***hair', 'head***hair***sideburns',
            'head***jaw', 'head***moustache', 'head***mouth***inferior lip', 'head***mouth***oral comisure',
            'head***mouth***superior lip', 'head***mouth***teeth', 'head***neck', 'head***nose',
            'head***nose***ala of nose', 'head***nose***bridge', 'head***nose***nose tip', 'head***nose***nostril',
            'head***philtrum', 'head***temple', 'head***wrinkles'
        ]

    def __getitem__(self, index):
        item = super().__getitem__(index)
        if hasattr(item, 'ancestry'):
            call_idx = len(self.selections)
            region_name = self.non_bg_classes[call_idx] if call_idx < len(self.non_bg_classes) else f"Region {call_idx+1:02d}"
            self.selections.append((region_name, item.ancestry))
        return item


def brdas_sampler(father_pool, mother_pool, father_weight=0.5, mother_weight=0.5):
    """
    Balanced Region-wise Dual-Ancestry Sampling (BRDAS) Sampler.
    For each of the 33 non-background region facial genes (RFGs):
      - Flips a coin using configured parent weights (default 0.5 / 0.5) to decide ancestry.
      - Samples one mutation vector from the selected pool.
      - Falls back gracefully to the other parent pool if one is empty.
    Returns a BrdasList wrapper to track region-wise ancestry selections.
    """
    num_regions = 33
    sampled_items = []
    
    total_w = father_weight + mother_weight
    father_p = father_weight / total_w if total_w > 0 else 0.5
    
    for _ in range(num_regions):
        if not father_pool and not mother_pool:
            break
        elif not father_pool:
            selected_pool = mother_pool
            ancestry = "Mother"
        elif not mother_pool:
            selected_pool = father_pool
            ancestry = "Father"
        else:
            if random.random() < father_p:
                selected_pool = father_pool
                ancestry = "Father"
            else:
                selected_pool = mother_pool
                ancestry = "Mother"
                
        if selected_pool:
            mu, var = random.choice(selected_pool)
            sampled_items.append(AncestryTuple(mu, var, ancestry))
            
    return BrdasList(sampled_items)


def init_model(image_size=1024, latent_dim=512):
    ckp = torch.load(path_ckpt_e4e, map_location="cpu", weights_only=False)
    encoder = Encoder4Editing(50, "ir_se", image_size).eval()
    encoder.load_state_dict(get_keys(ckp, "encoder"), strict=True)
    mean_latent = ckp["latent_avg"].to("cpu")
    mean_latent.unsqueeze_(0)

    generator = Generator(image_size, latent_dim, 8)
    checkpoint = torch.load(path_ckpt_stylegan2, map_location="cpu", weights_only=False)
    generator.load_state_dict(checkpoint["g_ema"], strict=False)
    generator.eval()

    sub2w   = MappingSub2W(N=18).eval()
    w2sub34 = MappingW2Sub(N=18).eval()
    ckp_sg  = torch.load(path_ckpt_stylegene, map_location="cpu", weights_only=False)
    w2sub34.load_state_dict(get_keys(ckp_sg, "w2sub34"))
    sub2w.load_state_dict(get_keys(ckp_sg, "sub2w"))

    requires_grad(sub2w, False)
    requires_grad(w2sub34, False)
    requires_grad(encoder, False)
    requires_grad(generator, False)
    return encoder, generator, sub2w, w2sub34, mean_latent


def tensor2rgb(tensor):
    tensor = (tensor * 0.5 + 0.5) * 255
    tensor = torch.clip(tensor, 0, 255).squeeze(0)
    tensor = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    return tensor.astype(np.uint8)


def generate_child(w18_F, w18_M, random_fakes, gamma=0.47, eta=0.4, arcs_lambda=0.0, sensitivity_map=None,
                   child_gender='male', geometry_weight=0.7, texture_weight=0.5):
    w18_syn = fuse_latent(w2sub34, sub2w, w18_F=w18_F, w18_M=w18_M,
                          random_fakes=random_fakes, fixed_gamma=gamma, fixed_eta=eta, arcs_lambda=arcs_lambda,
                          sensitivity_map=sensitivity_map, child_gender=child_gender,
                          geometry_weight=geometry_weight, texture_weight=texture_weight)
    img_C, _ = generator([w18_syn], return_latents=True, input_is_latent=True)
    return img_C, w18_syn
'''

# Find all gene_crossover_mutation.py paths in the current directory tree
crossover_paths = []
for root, dirs, files in os.walk('.'):
    if os.path.basename(root) == 'stylegene' and 'gene_crossover_mutation.py' in files:
        crossover_paths.append(os.path.join(root, 'gene_crossover_mutation.py'))

if not crossover_paths:
    crossover_paths = ['/kaggle/working/StyleGene/models/stylegene/gene_crossover_mutation.py']

for path in crossover_paths:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(new_crossover)
    print(f'✅ gene_crossover_mutation.py rewritten at: {path}')

# Find all api.py paths in the current directory tree to handle nested/duplicate clones
api_paths = []
for root, dirs, files in os.walk('.'):
    if os.path.basename(root) == 'stylegene' and 'api.py' in files:
        api_paths.append(os.path.join(root, 'api.py'))

if not api_paths:
    api_paths = ['/kaggle/working/StyleGene/models/stylegene/api.py']

for path in api_paths:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(new_api)
    print(f'✅ api.py rewritten at: {path}')


✅ gene_crossover_mutation.py rewritten at: ./models/stylegene/gene_crossover_mutation.py
✅ api.py rewritten at: ./models/stylegene/api.py


In [6]:
import sys

if "models.stylegene.api" in sys.modules:
    del sys.modules["models.stylegene.api"]
if "models.stylegene.gene_crossover_mutation" in sys.modules:
    del sys.modules["models.stylegene.gene_crossover_mutation"]

import models.stylegene.api as api

print("Imported successfully")

Imported successfully


## 6. Initialization of Models and Gene Pool
Loading the StyleGAN2 generator, e4e encoder, and initializing the custom GenePoolFactory to handle the multi-ethnic demographic splits.

In [7]:
import importlib
import models.stylegene.api as api_module

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load models
encoder, generator, sub2w, w2sub34, mean_latent = api_module.init_model()
encoder     = encoder.to(device)
generator   = generator.to(device)
sub2w       = sub2w.to(device)
w2sub34     = w2sub34.to(device)
mean_latent = mean_latent.to(device)
print(f"✅ Models loaded | mean_latent std: {mean_latent.std():.4f}")

# Inject into api module so generate_child() can use them
api_module.generator = generator
api_module.w2sub34   = w2sub34
api_module.sub2w     = sub2w

from models.stylegene.api import generate_child, tensor2rgb
print("✅ generate_child ready")

Device: cuda:0
✅ Models loaded | mean_latent std: 0.1542
✅ generate_child ready


In [8]:
import pickle
from models.stylegene.gene_pool import GenePoolFactory
from configs import path_ckpt_genepool

# Load with pickle directly — torch.load fails on large pickle4 files
print("Loading gene pool (8.11GB — takes ~60 seconds)...")
print("Loading pool...")
pool_data = torch.load(path_ckpt_genepool, 
                        map_location='cpu', weights_only=False)
print(f"✅ Loaded: {len(pool_data)} keys")

print(f"Pool type: {type(pool_data)}")
print(f"Pool keys: {len(pool_data) if hasattr(pool_data, '__len__') else 'N/A'}")

# Check if it's the right format
if isinstance(pool_data, dict):
    sample_key = list(pool_data.keys())[0]
    print(f"Sample key: {sample_key}")
    sample_val = pool_data[sample_key]
    print(f"Sample value type: {type(sample_val)}")
    if isinstance(sample_val, list) and len(sample_val) > 0:
        mu, var = sample_val[0]
        print(f"  mu shape: {mu.shape}  var shape: {var.shape}")
    print(f"\nAll keys:")
    for k in sorted(pool_data.keys()):
        print(f"  {k}: {len(pool_data[k])} samples")

Loading gene pool (8.11GB — takes ~60 seconds)...
Loading pool...
✅ Loaded: 56 keys
Pool type: <class 'dict'>
Pool keys: 56
Sample key: 0-2-male-White
Sample value type: <class 'list'>
  mu shape: torch.Size([1, 18, 34, 512])  var shape: torch.Size([1, 18, 34, 512])

All keys:
  0-2-female-Black: 1 samples
  0-2-female-East Asian: 38 samples
  0-2-female-Indian: 1 samples
  0-2-female-Latino_Hispanic: 19 samples
  0-2-female-Middle Eastern: 2 samples
  0-2-female-Southeast Asian: 21 samples
  0-2-female-White: 100 samples
  0-2-male-Black: 10 samples
  0-2-male-East Asian: 73 samples
  0-2-male-Indian: 5 samples
  0-2-male-Latino_Hispanic: 18 samples
  0-2-male-Middle Eastern: 19 samples
  0-2-male-Southeast Asian: 36 samples
  0-2-male-White: 100 samples
  10-19-female-Black: 64 samples
  10-19-female-East Asian: 86 samples
  10-19-female-Indian: 36 samples
  10-19-female-Latino_Hispanic: 19 samples
  10-19-female-Middle Eastern: 2 samples
  10-19-female-Southeast Asian: 69 samples
  

In [9]:
from models.stylegene.fair_face_model import init_fair_model, predict_race

geneFactor = GenePoolFactory(
    root_ffhq   = None,
    device      = device,
    mean_latent = mean_latent,
    max_sample  = 300
)
geneFactor.pools = pool_data
print(f"✅ Gene pool ready — {len(geneFactor.pools)} keys")

# FairFace
model_fair_7 = init_fair_model(device)
print("✅ FairFace ready")

✅ Gene pool ready — 56 keys


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ FairFace ready


## 7. Input Data Initialization & Reconstruction Test
Loading the 7 locked evaluation pairs. A quick inversion reconstruction is performed to verify the e4e encoder's baseline accuracy before genetic mixing.

In [11]:
!find /kaggle/input -iname "*.jpg" | head -100

/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/child_p2.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p5.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p7.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p1.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p4.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p3.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p7.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/child_p6.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p4.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p1.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p6.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p6.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p2.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p5.jpg
/kaggle/input/datasets/manaswimendhe

In [12]:
import os, tempfile, cv2
import numpy as np
import torch
from PIL import Image
from models.stylegene.util import load_img

PHOTOS = '/kaggle/input/datasets/izpiz06/locked-7-pairs'

# Import align_face only after confirming .dat file exists
assert os.path.exists('/tmp/ckpt/shape_predictor_68_face_landmarks.dat'), \
    "Landmark .dat file missing — re-run Cell 3"
print(f"✅ Landmark file: {os.path.getsize('/tmp/ckpt/shape_predictor_68_face_landmarks.dat')/1e6:.1f} MB")

from preprocess.align_images import align_face

def encode_face(img_path):
    raw     = np.array(Image.open(img_path).convert('RGB'))
    aligned = align_face(raw)
    tmp     = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    cv2.imwrite(tmp.name, cv2.cvtColor(aligned, cv2.COLOR_RGB2BGR))
    img_t   = load_img(tmp.name).to(device)
    w18     = encoder(img_t) + mean_latent
    return img_t, w18

# Test on father p1
img_t_f, w_f = encode_face(f'{PHOTOS}/father_p1.jpg')
with torch.no_grad():
    recon, _ = generator([w_f], input_is_latent=True, return_latents=True)
    Image.fromarray(tensor2rgb(recon)).save('./recon_test.png')
print(f"w_f std: {w_f.std():.4f}  (should be > 0.2)")
print("✅ Saved recon_test.png")

✅ Landmark file: 99.7 MB


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/izpiz06/locked-7-pairs/father_p1.jpg'

## 8. Age-Progressive Inference Pipeline
The core generation loop. This function processes the parent latents, applies the custom crossover/mutation algorithms, and generates the consistent child face at ages 5-10, 11-15, and 16-21.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tempfile, cv2

POOL_AGE_MAP = {
    '5-10':  '3-9',
    '11-15': '10-19',
    '16-21': '20-29'
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def query_parent_pools(pool_age, gender, race_f, race_m):
    """
    Retrieve parent pools independently.
    If father_race == mother_race, return a single pool list (compatibility mode).
    If father_race != mother_race, return a dict containing both pools.
    Apply age-expansion fallback independently to each pool if it is empty.
    """
    if race_f == race_m:
        entries = geneFactor(encoder, w2sub34, pool_age, gender, race_f)
        if not entries:
            print(f"  ⚠️ Same-race pool empty for {pool_age}-{gender}-{race_f} — expanding age bucket")
            for age in ['0-2', '3-9', '10-19', '20-29']:
                if age != pool_age:
                    entries += geneFactor(encoder, w2sub34, age, gender, race_f)
        return entries

    # Retrieve two independent pools
    father_pool = geneFactor(encoder, w2sub34, pool_age, gender, race_f)
    mother_pool = geneFactor(encoder, w2sub34, pool_age, gender, race_m)

    # Step 5: Handle sparse demographic buckets (age expansion)
    if not father_pool:
        print(f"  ⚠️ Father pool empty for {pool_age}-{gender}-{race_f} — expanding age bucket")
        expanded = []
        for age in ['0-2', '3-9', '10-19', '20-29']:
            expanded += geneFactor(encoder, w2sub34, age, gender, race_f)
        father_pool = expanded

    if not mother_pool:
        print(f"  ⚠️ Mother pool empty for {pool_age}-{gender}-{race_m} — expanding age bucket")
        expanded = []
        for age in ['0-2', '3-9', '10-19', '20-29']:
            expanded += geneFactor(encoder, w2sub34, age, gender, race_m)
        mother_pool = expanded

    return {
        "father_pool": father_pool,
        "mother_pool": mother_pool
    }

def full_pipeline(path_father, path_mother,
                  gender='male', child_seed=42,
                  race_f=None, race_m=None,
                  gamma=0.47, eta=0.4,
                  father_weight=0.5, mother_weight=0.5,
                  arcs_lambda=0.0):

    # ── Step 1: Align + encode parents ───────────────────
    raw_f = np.array(Image.open(path_father).convert('RGB'))
    raw_m = np.array(Image.open(path_mother).convert('RGB'))

    aligned_f = align_face(raw_f)
    aligned_m = align_face(raw_m)

    tmp_f = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    tmp_m = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    cv2.imwrite(tmp_f.name, cv2.cvtColor(aligned_f, cv2.COLOR_RGB2BGR))
    cv2.imwrite(tmp_m.name, cv2.cvtColor(aligned_m, cv2.COLOR_RGB2BGR))

    img_f = load_img(tmp_f.name).to(device)
    img_m = load_img(tmp_m.name).to(device)

    w18_F = encoder(F.interpolate(img_f, size=(256, 256))) + mean_latent
    w18_M = encoder(F.interpolate(img_m, size=(256, 256))) + mean_latent

    # ── Step 2: Race detection (auto, or override) ───────
    if race_f is None:
        race_f, _, _, _ = predict_race(model_fair_7, img_f.clone(), device)
    if race_m is None:
        race_m, _, _, _ = predict_race(model_fair_7, img_m.clone(), device)
    print(f"  Father race: {race_f} | Mother race: {race_m}")

    # ── Step 3: Loop over 3 age buckets ──────────────────
    results = {}
    brdas_logs = {}
    from models.stylegene.api import brdas_sampler, BrdasList

    for display_age, pool_age in POOL_AGE_MAP.items():
        set_seed(child_seed)  # frozen DNA — same crossover weights every age

        # Query pools separately
        pools = query_parent_pools(pool_age, gender, race_f, race_m)

        if isinstance(pools, dict):
            # mixed race: run BRDAS sampler
            random_fakes = brdas_sampler(
                pools["father_pool"], pools["mother_pool"],
                father_weight=father_weight, mother_weight=mother_weight
            )
        else:
            # same race: use original pool directly
            random_fakes = pools

        # Crossover + mutation + layer fusion
        img_C, _ = generate_child(
            w18_F.clone(), w18_M.clone(),
            random_fakes,
            gamma=gamma,
            eta=eta,
            arcs_lambda=arcs_lambda,
            child_gender=gender
        )
        results[display_age] = tensor2rgb(img_C)
        print(f"  ✓ {display_age} done")
        
        # Log selection statistics
        if isinstance(random_fakes, BrdasList):
            selections = list(random_fakes.selections)
            brdas_logs[display_age] = selections
            
            f_count = sum(1 for _, a in selections if a == "Father")
            m_count = sum(1 for _, a in selections if a == "Mother")
            print(f"    [BRDAS Log] {display_age} | Father regions: {f_count} | Mother regions: {m_count}")
            # Detailed console print of region assignments
            for reg, anc in selections:
                print(f"      {reg} ──> {anc}")

    if brdas_logs:
        results['brdas_logs'] = brdas_logs
        
    # ── Step 4: Write logs to a separate file ────────────
    import datetime
    log_file_path = './brdas_generation.log'
    try:
        with open(log_file_path, 'a', encoding='utf-8') as lf:
            lf.write("==================================================\n")
            lf.write(f"Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            lf.write(f"Parents: Father Race ({race_f}) x Mother Race ({race_m})\n")
            lf.write(f"Child Config: Gender={gender}, Seed={child_seed}\n")
            if brdas_logs:
                for age, selections in brdas_logs.items():
                    f_count = sum(1 for _, a in selections if a == "Father")
                    m_count = sum(1 for _, a in selections if a == "Mother")
                    lf.write(f"  Age {age} | Father regions: {f_count} | Mother regions: {m_count}\n")
                    for reg, anc in selections:
                        lf.write(f"    - {reg}: {anc}\n")
            else:
                lf.write("  BRDAS Bypassed (Same-Race / Original Pipeline)\n")
            lf.write("==================================================\n\n")
    except Exception as e:
        print(f"  ⚠️ Failed to write to log file: {e}")

    return results

def display_results(img_f, img_m, results, title='', real_child=None):
    ncols = 6 if real_child is not None else 5
    fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    items = [
        (img_f,              'Father'),
        (img_m,              'Mother'),
        (results['5-10'],    'Child 5-10'),
        (results['11-15'],   'Child 11-15'),
        (results['16-21'],   'Child 16-21'),
    ]
    if real_child is not None:
        items.append((real_child, 'Real Child'))
    for ax, (img, ttl) in zip(axes, items):
        ax.imshow(img)
        ax.set_title(ttl, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(
        f'./{title.replace(" ", "_").replace("+", "_")}.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()
    print(f"Saved {title}")

# ── Run p1 as smoke test ──────────────────────────────────
img_f_np = np.array(Image.open(f'{PHOTOS}/father_p1.jpg').convert('RGB'))
img_m_np = np.array(Image.open(f'{PHOTOS}/mother_p1.jpg').convert('RGB'))

results = full_pipeline(
    path_father = f'{PHOTOS}/father_p1.jpg',
    path_mother = f'{PHOTOS}/mother_p1.jpg',
    gender      = 'male',
    child_seed  = 42,
    race_f      = 'Indian',
    race_m      = 'Indian',
    gamma       = 0.05,
    eta         = 0.4
)

real_child_p1 = np.array(Image.open(f'{PHOTOS}/child_p1.png').convert('RGB'))

display_results(
    img_f_np, img_m_np, results,
    title='P1 Shahrukh + Gauri',
    real_child=real_child_p1
)



## 9. Exploratory Generation & Seed Testing
Executing initial generation runs across the test pairs. These baseline outputs are used to analyze seed variance and select the optimal latent parameters that maximize age progression while preserving identity.

In [ ]:
OUT_DIR = './outputs_v2_stylegene'
os.makedirs(OUT_DIR, exist_ok=True)

pairs_config = [
    ('p1_shahrukh_gauri', f'{PHOTOS}/father_p1.jpg', f'{PHOTOS}/mother_p1.jpg',  'male',            'Indian',         'Indian',         f'{PHOTOS}/child_p1.png'),
    ('p2_jackie_joan',    f'{PHOTOS}/father_p2.jpg', f'{PHOTOS}/mother_p2.jpeg', 'male',            'East Asian',     'East Asian',     f'{PHOTOS}/child_p2.jpg'),
    ('p3_obama_michelle', f'{PHOTOS}/father_p3.jpg', f'{PHOTOS}/mother_p3.jpeg', 'female',          'Black',          'Black',          f'{PHOTOS}/child_p3.jpg'),
    ('p4_tomhanks_rita',  f'{PHOTOS}/father_p4.jpg', f'{PHOTOS}/mother_p4.jpg',  'male',            'White',          'White',          f'{PHOTOS}/child_p4.jpg'),
    ('p5_ben_laura',      f'{PHOTOS}/father_p5.jpg', f'{PHOTOS}/mother_p5.jpg',  'female',          'Black',          'White',          f'{PHOTOS}/child_p5.jpg'),
    ('p6_tiger_elin',     f'{PHOTOS}/father_p6.jpg', f'{PHOTOS}/mother_p6.jpg',  'female',          'Black',          'White',          f'{PHOTOS}/child_p6.jpg'),
    ('p7_mark_kelly',     f'{PHOTOS}/father_p7.jpg', f'{PHOTOS}/mother_p7.jpg',  'female',          'Latino_Hispanic','White',          f'{PHOTOS}/child_p7.jpg'),
]

for name, pf, pm, gender, race_f, race_m, child_path in pairs_config:
    print(f"\n{'='*50}")
    print(f"Running: {name}")
    try:
        # full_pipeline returns ONLY the results dict — do NOT unpack as 3 values
        results = full_pipeline(
            path_father = pf,
            path_mother = pm,
            gender      = gender,
            race_f      = race_f,
            race_m      = race_m,
            child_seed  = 42,
            gamma       = 0.47,
            eta         = 0.4,
        )

        # Save each age bucket — filter by image type to skip metadata (e.g., brdas_logs)
        for bucket, img_np in results.items():
            if not isinstance(img_np, (np.ndarray, Image.Image)):
                continue
            out_path = f'{OUT_DIR}/{name}_{bucket}.png'
            Image.fromarray(img_np).save(out_path)
            print(f"  💾 Saved: {out_path}")

        # Load parent images for display (separately — not from pipeline return)
        img_f_np = np.array(Image.open(pf).convert('RGB'))
        img_m_np = np.array(Image.open(pm).convert('RGB'))
        real_child = np.array(Image.open(child_path).convert('RGB')) if os.path.exists(child_path) else None

        display_results(img_f_np, img_m_np, results, title=name, real_child=real_child)
        print(f"✅ {name} complete")

    except Exception as e:
        print(f"❌ {name} FAILED: {e}")
        import traceback; traceback.print_exc()

## 10. Final Generation (Optimal Seeds Locked)
Running the definitive inference pass using the best-performing seeds identified in the testing phase. These final outputs are saved to a dedicated directory and permanently locked for quantitative evaluation and grid visualization.

In [ ]:
OUT_DIR = './outputs_final'
os.makedirs(OUT_DIR, exist_ok=True)

BEST_SEEDS = {
    'p1_shahrukh_gauri':  256,
    'p2_jackie_joan':     256,
    'p3_obama_michelle':  42,
    'p4_tomhanks_rita':   256,
    'p5_ben_laura':       256,
    'p6_tiger_elin':      42,
    'p7_mark_kelly':      42,
}

pairs_config_final = [
    ('p1_shahrukh_gauri', f'{PHOTOS}/father_p1.jpg',  f'{PHOTOS}/mother_p1.jpg',  'male',   'Indian',          'Indian',     f'{PHOTOS}/child_p1.png'),
    ('p2_jackie_joan',    f'{PHOTOS}/father_p2.jpg',  f'{PHOTOS}/mother_p2.jpeg', 'male',   'East Asian',      'East Asian', f'{PHOTOS}/child_p2.jpg'),
    ('p3_obama_michelle', f'{PHOTOS}/father_p3.jpg',  f'{PHOTOS}/mother_p3.jpeg', 'female', 'Black',           'Black',      f'{PHOTOS}/child_p3.jpg'),
    ('p4_tomhanks_rita',  f'{PHOTOS}/father_p4.jpg',  f'{PHOTOS}/mother_p4.jpg',  'male',   'White',           'White',      f'{PHOTOS}/child_p4.jpg'),
    ('p5_ben_laura',      f'{PHOTOS}/father_p5.jpg',  f'{PHOTOS}/mother_p5.jpg',  'female', 'Black',           'White',      f'{PHOTOS}/child_p5.jpg'),
    ('p6_tiger_elin',     f'{PHOTOS}/father_p6.jpg',  f'{PHOTOS}/mother_p6.jpg',  'female', 'Black',           'White',      f'{PHOTOS}/child_p6.jpg'),
    ('p7_mark_kelly',     f'{PHOTOS}/father_p7.jpg',  f'{PHOTOS}/mother_p7.jpg',  'female', 'Latino_Hispanic', 'White',      f'{PHOTOS}/child_p7.jpg'),
]

final_outputs = {}

for name, pf, pm, gender, race_f, race_m, child_path in pairs_config_final:
    print(f"\n{'='*50}")
    print(f"Running: {name} | seed={BEST_SEEDS[name]}")
    try:
        # full_pipeline now returns ONLY the results dict
        results = full_pipeline(
            path_father = pf,
            path_mother = pm,
            gender      = gender,
            race_f      = race_f,
            race_m      = race_m,
            child_seed  = BEST_SEEDS[name],
            gamma       = 0.47,
            eta         = 0.4
        )

        # Load parent images separately for display and saving
        img_f = np.array(Image.open(pf).convert('RGB'))
        img_m = np.array(Image.open(pm).convert('RGB'))

        # Save each age bucket — filter by image type to skip metadata (e.g., brdas_logs)
        for bucket, img_np in results.items():
            if not isinstance(img_np, (np.ndarray, Image.Image)):
                continue
            out_path = f'{OUT_DIR}/{name}_{bucket}.png'
            Image.fromarray(img_np).save(out_path)

        # Save parent images
        Image.fromarray(img_f).save(f'{OUT_DIR}/{name}_father.png')
        Image.fromarray(img_m).save(f'{OUT_DIR}/{name}_mother.png')

        final_outputs[name] = results

        real_child = np.array(Image.open(child_path).convert('RGB')) if os.path.exists(child_path) else None
        display_results(img_f, img_m, results, title=name, real_child=real_child)
        print(f"✅ {name} saved")

    except Exception as e:
        print(f"❌ {name} FAILED: {e}")
        import traceback; traceback.print_exc()

print(f"\n✅ All outputs saved to {OUT_DIR}")

## 11. Quantitative Evaluation Setup & Execution
Defining and running the evaluation metrics:
* **LPIPS:** Measures perceptual variance across the age stages.
* **SSIM:** Measures structural similarity against the real ground-truth child.
* **ArcFace:** Measures identity preservation across the 3 age outputs.

In [ ]:
!pip install lpips facenet-pytorch deepface -q

import torch
import numpy as np
import os
from PIL import Image
from torchvision import transforms
import lpips
import cv2

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ── 1. LPIPS — measures age diversity between 5-10 and 16-21 ─────────
# Higher = more visible age difference = better
loss_fn_lpips = lpips.LPIPS(net='alex').to(device)

def compute_lpips(img1_pil, img2_pil):
    T = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    t1 = T(img1_pil.convert('RGB')).unsqueeze(0).to(device)
    t2 = T(img2_pil.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        return loss_fn_lpips(t1, t2).item()

# ── 2. SSIM — similarity between generated child and real child ───────
from skimage.metrics import structural_similarity as ssim_fn

def compute_ssim(img1_pil, img2_pil):
    img1 = cv2.resize(np.array(img1_pil.convert('RGB')), (256, 256))
    img2 = cv2.resize(np.array(img2_pil.convert('RGB')), (256, 256))
    return ssim_fn(img1, img2, channel_axis=2, data_range=255)

# ── 3. ArcFace Kinship Similarity ────────────────────────────────────
# Measures if generated child looks genetically related to parents
from facenet_pytorch import InceptionResnetV1

arcface = InceptionResnetV1(pretrained='vggface2').to(device).eval()

T_arc = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

def get_embedding(img_pil):
    t = T_arc(img_pil.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = arcface(t)
    return emb / emb.norm()

def compute_kinship_similarity(parent1_pil, parent2_pil, child_pil):
    emb_f = get_embedding(parent1_pil)
    emb_m = get_embedding(parent2_pil)
    emb_c = get_embedding(child_pil)
    emb_p = (emb_f + emb_m) / 2
    return torch.nn.functional.cosine_similarity(emb_p, emb_c).item()

# ── 4. DeepFace Age Estimation ────────────────────────────────────────
# Trained specifically on face images — accurate age prediction
from deepface import DeepFace

AGE_RANGES = {
    '5-10':  (5,  10),
    '11-15': (11, 15),
    '16-21': (16, 21),
}

def predict_age_deepface(img_pil):
    try:
        # Save to temp file for DeepFace
        tmp_path = '/tmp/tmp_age_eval.png'
        img_pil.save(tmp_path)
        result = DeepFace.analyze(
            tmp_path,
            actions    = ['age'],
            enforce_detection = False,
            silent     = True
        )
        return result[0]['age']
    except Exception as e:
        print(f"    DeepFace error: {e}")
        return None

def age_in_range(predicted_age, bucket):
    if predicted_age is None:
        return False
    low, high = AGE_RANGES[bucket]
    return low <= predicted_age <= high

print("✅ All metric functions ready")

In [ ]:
# ── Run metrics on all 7 pairs ────────────────────────────────────────
import os
import numpy as np
import pandas as pd
from PIL import Image

OUT_DIR = './outputs_final'
PHOTOS  = '/kaggle/input/datasets/izpiz06/locked-7-pairs'

PAIRS = [
    ("p1_shahrukh_gauri",  "father_p1.jpg",  "mother_p1.jpg",  "child_p1.png"),
    ("p2_jackie_joan",     "father_p2.jpg",  "mother_p2.jpeg", "child_p2.jpg"),
    ("p3_obama_michelle",  "father_p3.jpg",  "mother_p3.jpeg", "child_p3.jpg"),
    ("p4_tomhanks_rita",   "father_p4.jpg",  "mother_p4.jpg",  "child_p4.jpg"),
    ("p5_ben_laura",       "father_p5.jpg",  "mother_p5.jpg",  "child_p5.jpg"),
    ("p6_tiger_elin",      "father_p6.jpg",  "mother_p6.jpg",  "child_p6.jpg"),
    ("p7_mark_kelly",      "father_p7.jpg",  "mother_p7.jpg",  "child_p7.jpg"),
]
BUCKETS = ['5-10', '11-15', '16-21']

results = []

for pid, fname, mname, cname in PAIRS:
    real_path = f'{PHOTOS}/{cname}'
    has_real  = os.path.exists(real_path)
    img_real  = Image.open(real_path).convert('RGB') if has_real else None

    # Load all 3 generated buckets
    gen_imgs = {}
    for bucket in BUCKETS:
        gen_path = f'{OUT_DIR}/{pid}_{bucket}.png'
        if os.path.exists(gen_path):
            gen_imgs[bucket] = Image.open(gen_path).convert('RGB')
        else:
            print(f"  ⚠️  Missing: {pid}_{bucket}.png")

    # LPIPS age progression — youngest vs oldest bucket
    lpips_diversity = None
    if '5-10' in gen_imgs and '16-21' in gen_imgs:
        lpips_diversity = compute_lpips(gen_imgs['5-10'], gen_imgs['16-21'])

    for bucket in BUCKETS:
        if bucket not in gen_imgs:
            continue
        img_gen = gen_imgs[bucket]

        if has_real:
            img_real_np = np.array(img_real.resize((256, 256)))
            img_gen_np  = np.array(img_gen.resize((256, 256)))
            ssim_score  = ssim_fn(img_real_np, img_gen_np, channel_axis=2, data_range=255)
        else:
            ssim_score = None
        results.append({
            'pair':            pid,
            'bucket':          bucket,
            'ssim':            round(ssim_score,     4) if ssim_score     is not None else None,
            'lpips_diversity': round(lpips_diversity, 4) if lpips_diversity is not None else None,
        })

        print(f"  {pid} {bucket} | "
              f"SSIM={f'{ssim_score:.3f}' if ssim_score is not None else 'N/A'}")

    print(f"  LPIPS Age Progression (5-10 vs 16-21): "
          f"{f'{lpips_diversity:.3f}' if lpips_diversity is not None else 'N/A'}")

# ── Summary ───────────────────────────────────────────────────────────
df = pd.DataFrame(results)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)

df['ssim'] = pd.to_numeric(df['ssim'], errors='coerce')
df['lpips_diversity'] = pd.to_numeric(df['lpips_diversity'], errors='coerce')

print(f"Mean SSIM:                  {df['ssim'].mean():.4f}")
print(f"Mean LPIPS Age Progression: {df['lpips_diversity'].mean():.4f}")

print("\nPer-pair SSIM (mean across 3 buckets):")
print(df.groupby('pair')['ssim'].mean().round(4))

print("\nPer-pair LPIPS Age Progression (5-10 vs 16-21):")
print(df.groupby('pair')['lpips_diversity'].first().round(4))

df.to_csv('./metrics_results.csv', index=False)
print("\n✅ Saved metrics_results.csv")

In [ ]:
# Gradio UI removed in repo cleanup
# The child_face_gradio_ui.py file was removed from this repository.
# If you need the Gradio demo, see the commit history before 1363e72,
# or reconstruct it from the pipeline cells above.
print("Gradio UI: see commit history pre-1363e72 or rebuild from pipeline cells.")

In [ ]:
# Gradio demo removed
# This cell previously loaded child_face_gradio_ui.py and launched a Gradio interface.
# That file has been removed from the repository during cleanup.
# Use the pipeline cells directly for inference.
print("Gradio demo removed. Run pipeline cells above for inference.")

In [ ]:
import os
import torch
from PIL import Image
import tqdm

# --- Windows Path Configuration ---
# Update this to where your harvest script saved the pictures!
input_base_dir = '/kaggle/input/datasets/izpiz06/dataset-upgrade/dataset_upgrade'        
existing_pool_path = '/kaggle/input/datasets/manaswimendhekar/stylegene-balanced-pool/pool_50samples.pkl' 
output_pool_path = '/kaggle/working/pool_fortified.pkl'   

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Loading existing gene pool...")
try:
    gene_pool = torch.load(existing_pool_path, map_location='cpu', weights_only=False)
except FileNotFoundError:
    print(f"Warning: {existing_pool_path} not found. Creating a brand new dictionary pool.")
    gene_pool = {}

def process_single_image(image_path):
    aligned_path = align_face(image_path)

    aligned_pil = Image.open(aligned_path).convert('RGB')

    aligned_np = np.array(aligned_pil)

    img_tensor = torch.from_numpy(aligned_np).permute(2, 0, 1).float()

    img_tensor = (img_tensor / 127.5) - 1.0
    img_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        w_latent = encoder(img_tensor) + mean_latent.to(device)
        mu, sigma, _ = w2sub34(w_latent)

    return mu.cpu(), sigma.cpu()

print("\nStarting batch gene extraction...")

for bucket_name in os.listdir(input_base_dir):
    bucket_folder = os.path.join(input_base_dir, bucket_name)
    if not os.path.isdir(bucket_folder):
        continue
        
    images = [f for f in os.listdir(bucket_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if not images:
        continue
        
    print(f"\nProcessing bucket [{bucket_name}] | Found {len(images)} images.")
    
    if bucket_name not in gene_pool:
        gene_pool[bucket_name] = []
        
    for img_file in tqdm.tqdm(images, desc=f"Extracting {bucket_name}"):
        img_path = os.path.join(bucket_folder, img_file)
        try:
            mu_tensor, sigma_tensor = process_single_image(img_path)
            gene_pool[bucket_name].append({'mu': mu_tensor, 'var': sigma_tensor})
        except Exception as e:
            print(f" -> Skipped {img_file} due to error: {e}")
            continue

print("\nSaving updated gene pool...")
torch.save(gene_pool, output_pool_path)
print(f"Success! Fortified pool saved to: {output_pool_path}")

In [ ]:
img_path = "/kaggle/input/datasets/izpiz06/dataset-upgrade/dataset_upgrade/0-2-female-Indian/" + os.listdir("/kaggle/input/datasets/izpiz06/dataset-upgrade/dataset_upgrade/0-2-female-Indian")[0]

mu, sigma = process_single_image(img_path)

print(mu.shape)
print(sigma.shape)